# BDM2/P1 Darcy: conservative high-order local flux

The MHM skeleton restricts a BDM2 normal trace while retaining the full interior flux moments. Pressure is discontinuous P1. The flux normal is conforming across fine and macro interfaces; tangential flux and pressure can remain discontinuous. This follows the mixed-local framework of [Durán et al. (2019)](https://doi.org/10.1016/j.cma.2019.05.013); the following are original analytical verifications, not reproduced table values.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/27804b11a4328712d86c8e34eb45da34f073341c40c9231dc4ebf76993638791/17_darcy_bdm-companion.zip"
COMPANION_SHA256 = "27804b11a4328712d86c8e34eb45da34f073341c40c9231dc4ebf76993638791"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/17_darcy_bdm.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from pymhm import TriangleMesh


## Quadratic pressure and exact linear flux

For `p=x²+y²`, permeability `K=[[2,.3],[.3,1]]`, and `f=-6`, the exact flux is `q=-2Kx`. BDM2 contains this flux exactly; the discrete P1 pressure equals the cellwise L2 projection and therefore has a nonzero error.


## Declare local and global forms

The application supplies the local mixed saddle for coefficients u=(q,p,r),
where r is the private boundary-pressure multiplier:

$$
\begin{aligned}
A_T&=\begin{bmatrix}M&-D^{\mathsf T}&N\\-D&0&0\\N^{\mathsf T}&0&0\end{bmatrix},
&f_T&=\begin{bmatrix}0\\-F\\0\end{bmatrix},\\
B_T&=\begin{bmatrix}0\\0\\-R\end{bmatrix},
&C_T&=-B_T^{\mathsf T}.
\end{aligned}
$$

Here M tests K inverse, D tests divergence, N selects exterior normal moments,
and R maps the fixed macroface flux orientation to outward fine-edge moments.
RT/BDM kernels supply their executed normal/interior basis; the pressure and
constant boundary-pressure modes form the declared joint kernel. Physical
pressure integrals provide its moments without volume normalization. The
global `Equation` supplies pressure boundary data; pure Neumann data add the
physical mean constraint. `assemble` executes these declarations and recovery
interprets the same H(div) flux and DG pressure coefficients.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.mixed_darcy import define_bdm_darcy, recover_bdm_darcy

tensor = np.array([[2.0, 0.3], [0.3, 1.0]])
solution_definition = define_bdm_darcy(
    TriangleMesh.unit_square(2),
    permeability=tensor,
    source=-6,
    dirichlet=lambda x: np.sum(x * x, axis=1),
)
solution_system = assemble(solution_definition.problem)
solution = recover_bdm_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
flux_error = solution.flux_l2_error(lambda x: -2 * x @ tensor)
assert flux_error < 3e-12
assert solution.divergence_l2_error(-6.0) < 3e-12
assert np.max(np.abs(solution.conservation_residuals())) < 3e-12
{"pressure_L2": solution.l2_error(lambda x: np.sum(x * x, axis=1)), "flux_L2": flux_error}


## Cosine field: enriching the trace separately

The archived case uses `p=cos(πx)cos(πy)`, identity permeability and the actual nonhomogeneous boundary trace. It fixes 32 diagonal macrotriangles and local refinement four while changing only the unsplit macro trace degree. A high-order local solver does not eliminate the error of an under-resolved constant normal trace.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    records = json.loads((ROOT / "examples/results/darcy-bdm.json").read_text())
    records["trace_study"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/darcy-bdm/cosine-fields.png")))
    display(Image(filename=str(ROOT / "docs/figures/darcy-bdm/convergence-and-trace.png")))


## Equilibrium and independent assembly

`fine_equilibrium_residuals()` returns all three P1 moments of `div(q)-f` per fine cell. `fine_conservation_residuals()` returns their sum, and `normal_flux_residuals()` checks the normal moments against the signed skeleton. For nonpolynomial f, these zero moments do not imply zero pointwise divergence error.

A separate native test compares all fields with a global DOLFINx/UFL and Basix BDM2/P1 assembly on the identical fine mesh. The macro trace is fully enriched for that discrete-space equivalence. No external comparison driver is needed to run this notebook.


In [ ]:
{"max_P1_equilibrium_moment": max(float(np.max(np.abs(v))) for v in solution.fine_equilibrium_residuals()),
 "max_normal_moment_defect": max(float(np.max(np.abs(v))) for v in solution.normal_flux_residuals())}
